# Spatial Models Demo: MNL, NestedMNL, MixedMNL, MixedNestedMNL with `graph=`

This notebook demonstrates the four spatial logit models in `locpick`. Each base estimator becomes a spatially correlated logit (SCL; Bhat & Guo 2004) when a spatial adjacency `graph=` is supplied:

- **`MNL(..., graph=g)`** — Spatially Correlated Logit
- **`NestedMNL(..., graph=g)`** — Spatial + nesting
- **`MixedMNL(..., graph=g)`** — Spatial + random taste variation
- **`MixedNestedMNL(..., graph=g)`** — Spatial + nesting + random variation

We use synthetic data generated by `simulate_scl` with a known spatial adjacency structure.

In [ ]:
import numpy as np
import pandas as pd

from locpick import ChoiceModel
from locpick.dgp import simulate_scl
from locpick.models.mixed import ParamDistribution
from locpick.models.nested import NestingTree, NestSpec

In [ ]:
import geosnap as gsp

In [ ]:
datasets = gsp.DataStore()

In [ ]:
dc = gsp.io.get_acs(datasets, years=2019, level="tract", state_fips="11")

In [ ]:
dc.plot()

In [ ]:
from libpysal.graph import Graph

dc_graph = Graph.build_contiguity(dc, rook=False)

adj = dc_graph.sparse.todense()

## 2. Generate Synthetic Data

We use `simulate_scl` to create a synthetic dataset with known spatial correlation. The DGP produces:
- `choosers`: household-level observations with income
- `alternatives`: tract-level attributes (cost, time)
- `adjacency`: a spatial adjacency matrix
- `true_rho`: the ground-truth spatial correlation parameter

In [ ]:
# Generate synthetic SCL data with spatial correlation
n_obs = 2000
n_alts = dc.shape[0]

scl_dataset = simulate_scl(
    n_obs=n_obs,
    n_alts=n_alts,
    alt_params={"cost": -0.5, "time": -0.2},
    rho=0.7,
    seed=42,
    adjacency=adj,
)

ct = scl_dataset.choice_table
print(f"Observations: {ct.n_observations}")
print(f"Alternatives: {ct.n_alternatives}")
print(f"True rho: {scl_dataset.true_rho}")
print(f"Adjacency shape: {scl_dataset.adjacency.shape}")

## 3. Configure Spatial Models

We configure four spatial model variants:

1. **`MNL(graph=g)`** — spatial correlation only
2. **`NestedMNL(graph=g)`** — spatial + nesting structure
3. **`MixedMNL(graph=g)`** — spatial + random taste variation
4. **`MixedNestedMNL(graph=g)`** — spatial + nesting + random variation

Each model shares the same `ChoiceTable` and formula but adds structural complexity. The `graph` parameter accepts a `libpysal.graph.Graph`, a `scipy.sparse` array, or a dense numpy adjacency matrix.

In [ ]:
# Common formula for all models
formula = "cost + time - 1"

# Use the adjacency matrix from the DGP directly
adj = scl_dataset.adjacency

# 1. Spatial MNL (SCL) — spatial correlation only
model_scl = ChoiceModel(
    ct,
    formula=formula,
    graph=adj,
)

# 2. Spatial NestedMNL — spatial + nesting
nest_tree = NestingTree(
    nests=[
        NestSpec(name="urban", alt_ids=list(range(0, n_alts // 2))),
        NestSpec(name="suburban", alt_ids=list(range(n_alts // 2, n_alts))),
    ]
)

model_nested_scl = ChoiceModel(
    ct,
    formula=formula,
    graph=adj,
    nests=nest_tree,
)

# 3. Spatial MixedMNL — spatial + random coefficients
random_params = {
    "time": ParamDistribution(distribution="normal", param="time"),
}

model_mixed_scl = ChoiceModel(
    ct,
    formula=formula,
    graph=adj,
    random_params=random_params,
    n_draws=100,
)

# 4. Spatial MixedNestedMNL — spatial + nesting + random coefficients
model_mixed_nested_scl = ChoiceModel(
    ct,
    formula=formula,
    graph=adj,
    nests=nest_tree,
    random_params=random_params,
    n_draws=100,
)

print("Models configured:")
print(f"  Spatial MNL:            {type(model_scl).__name__}")
print(f"  Spatial NestedMNL:      {type(model_nested_scl).__name__}")
print(f"  Spatial MixedMNL:       {type(model_mixed_scl).__name__}")
print(f"  Spatial MixedNestedMNL: {type(model_mixed_nested_scl).__name__}")

## 4. Fit Spatial Models

Fit each model and inspect the estimated coefficients. Spatial MNL estimates a `rho` parameter that captures spatial correlation. The nested variant adds `lambda` nest dissimilarity parameters. The mixed variant adds `sd_*` random-coefficient standard deviations.

In [ ]:
# Fit spatial MNL
result_scl = model_scl.fit()
print("=== Spatial MNL ===")
print(result_scl.summary())

In [ ]:
# Fit spatial NestedMNL
result_nested_scl = model_nested_scl.fit()
print("=== Spatial NestedMNL ===")
print(result_nested_scl.summary())

In [ ]:
# Fit spatial MixedMNL
result_mixed_scl = model_mixed_scl.fit()
print("=== Spatial MixedMNL ===")
print(result_mixed_scl.summary())

In [ ]:
# Fit spatial MixedNestedMNL
result_mixed_nested_scl = model_mixed_nested_scl.fit()
print("=== Spatial MixedNestedMNL ===")
print(result_mixed_nested_scl.summary())

## 5. Evaluate Model Performance

Compare log-likelihood, AIC, and BIC across the four spatial variants. More complex models should improve fit, but we can check whether the improvement justifies the additional parameters.

In [ ]:
# Collect fit statistics for comparison
results = {
    "Spatial MNL": result_scl,
    "Spatial NestedMNL": result_nested_scl,
    "Spatial MixedMNL": result_mixed_scl,
    "Spatial MixedNestedMNL": result_mixed_nested_scl,
}

comparison = pd.DataFrame(
    {
        name: {
            "Log-Likelihood": r.log_likelihood,
            "Null LL": r.log_likelihood_null,
            "AIC": r.aic,
            "BIC": r.bic,
            "Rho²": r.rho_squared,
            "Adj. Rho²": r.rho_bar_squared,
            "n_params": r.n_parameters,
        }
        for name, r in results.items()
    }
).T

print(comparison.round(4))

## 6. Generate Spatial Predictions

Predict choice probabilities for each model. The SCL family produces spatially correlated probabilities — nearby alternatives have more similar predicted shares than under MNL.

In [ ]:
# Predict probabilities using model.probabilities() (no arguments uses fitted parameters)
probs_scl = model_scl.probabilities()

print(f"Spatial MNL probabilities shape: {probs_scl.shape}")
print(f"Probabilities sum to 1: {np.allclose(probs_scl.sum(axis=1), 1.0)}")
print("\nFirst 3 decision-makers' probabilities:")
print(probs_scl[:3].round(4))

## 7. Visualize Spatial Results

Plot the spatial adjacency structure and predicted choice probabilities.

In [ ]:
import matplotlib.pyplot as plt

# Plot 1: Spatial adjacency matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Show the adjacency matrix
im = axes[0].imshow(adj[:20, :20], cmap="Blues", interpolation="nearest")
axes[0].set_title("Spatial Adjacency Matrix (first 20 alts)")
axes[0].set_xlabel("Alternative j")
axes[0].set_ylabel("Alternative i")
fig.colorbar(im, ax=axes[0])

# Plot 2: Predicted probabilities for a single decision-maker
axes[1].bar(range(n_alts), probs_scl[0], alpha=0.7, label="Spatial MNL")
axes[1].set_xlabel("Alternative")
axes[1].set_ylabel("Predicted Probability")
axes[1].set_title("Spatial MNL Predicted Probabilities (DM 0)")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
dc.assign(probs=probs_scl[0]).plot("probs", scheme="quantiles")